In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_data(path):
    """Read the CSV at `path` (handles a UTF-8 BOM) and drop the version marker."""
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    return df.drop(columns=["version_date"], errors="ignore")

df = load_data("../data/released_schedule.csv")
df.head()

In [ ]:
colonnes_requises = {"supplier_id", "plant", "schedule_quantity", "weight", "volume", "transmission_date"}
if manquantes := colonnes_requises - set(df.columns):
    raise ValueError(f"Colonnes absentes : {sorted(manquantes)}")

mesures = ["schedule_quantity", "weight", "volume"]
for colonne in mesures:
    convertie = pd.to_numeric(df[colonne], errors="coerce")
    invalides = (df[colonne].notna() & convertie.isna()).sum()
    if invalides:
        print(f"Attention : {invalides} valeurs non numériques dans {colonne}.")
    df[colonne] = convertie

for colonne in ["date", "transmission_date"]:
    df[colonne] = pd.to_datetime(df[colonne], errors="coerce")

print(f"{len(df):,} lignes · {df.shape[1]} colonnes analysées")

In [ ]:
profil = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "valeurs_distinctes": df.nunique(),
    "valeurs_manquantes": df.isna().sum(),
    "manquantes_%": df.isna().mean().mul(100).round(2),
})
profil["maximum_numérique"] = df.select_dtypes(include="number").max()

print("Features présentes — maximums calculés sur les colonnes numériques :")
display(profil)

In [ ]:
# Valeurs max des colonnes importantes pour le sujet (cf. docs/consolidation_algorithme_pdf.pdf).
capacite_camion = {"weight": 24_500, "volume": 110.4}  # kg, m³

maximums = df[mesures].max().rename("maximum")
display(maximums.to_frame())

for colonne, limite in capacite_camion.items():
    depassements = int((df[colonne] > limite).sum())
    print(f"Lignes où {colonne} dépasse la capacité d'un camion ({limite}) : {depassements}")

for colonne in ["date", "transmission_date"]:
    print(f"{colonne} : du {df[colonne].min().date()} au {df[colonne].max().date()}")

In [ ]:
resume_mesures = pd.DataFrame({
    "nombre_égal_à_0": df[mesures].eq(0).sum(),
    "pourcentage_égal_à_0": df[mesures].eq(0).mean().mul(100).round(2),
    "nombre_négatif": df[mesures].lt(0).sum(),
    "minimum": df[mesures].min(),
    "médiane": df[mesures].median(),
    "moyenne": df[mesures].mean(),
    "maximum": df[mesures].max(),
    "total": df[mesures].sum(min_count=1),
})
print("Quantités, poids et volumes :")
display(resume_mesures)
print(
    "Lignes où schedule_quantity, weight ET volume valent 0 :",
    int(df[mesures].eq(0).all(axis=1).sum()),
)

In [ ]:
def pair_counts(df, group, target):
    """Distinct `target` count per `group`, entities with no partner included at 0."""
    pairs = df[[group, target]].dropna().drop_duplicates()
    counts = pairs.groupby(group)[target].nunique()
    return counts.reindex(df[group].dropna().unique(), fill_value=0).sort_values(ascending=False)

suppliers_par_plant = pair_counts(df, "plant", "supplier_id")
plants_par_supplier = pair_counts(df, "supplier_id", "plant")
paires = df[["supplier_id", "plant"]].dropna().drop_duplicates()

print(f"Couples supplier–plant distincts et complets : {len(paires):,}")
print(f"Suppliers distincts : {df['supplier_id'].nunique():,}")
print(f"Plants distincts : {df['plant'].nunique():,}")
print(
    "Lignes avec supplier ou plant manquant :",
    int(df[["supplier_id", "plant"]].isna().any(axis=1).sum()),
)

print("\nNombre de suppliers distincts pour chaque plant :")
display(suppliers_par_plant.rename("nombre_suppliers").to_frame())
print("\nNombre de plants distincts pour chaque supplier :")
display(plants_par_supplier.rename("nombre_plants").to_frame())
print("\nStatistiques sur les relations :")
display(pd.DataFrame({
    "suppliers_par_plant": suppliers_par_plant.describe(),
    "plants_par_supplier": plants_par_supplier.describe(),
}))

In [ ]:
print("Lignes dupliquées sur les colonnes analysées (hors version_date) :", int(df.duplicated().sum()))
print("Lignes avec quantité = 0 et poids > 0 :",
      int((df["schedule_quantity"].eq(0) & df["weight"].gt(0)).sum()))
print("Lignes avec poids = 0 et quantité > 0 :",
      int((df["weight"].eq(0) & df["schedule_quantity"].gt(0)).sum()))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for serie, ax, titre, xlabel in [
    (resume_mesures["nombre_égal_à_0"], axes[0], "Valeurs égales à zéro", "Nombre de lignes"),
    (suppliers_par_plant, axes[1], "Suppliers distincts par plant — top 20", "Nombre de suppliers"),
    (plants_par_supplier, axes[2], "Plants distincts par supplier — top 20", "Nombre de plants"),
]:
    if serie.empty:
        ax.text(0.5, 0.5, "Aucune donnée", ha="center", va="center")
    else:
        serie.head(20).sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set(title=titre, xlabel=xlabel, ylabel="")
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, colonne in zip(axes, mesures):
    repartition = pd.Series({
        "Zéro": df[colonne].eq(0).sum(),
        "Non-zéro": (df[colonne].notna() & df[colonne].ne(0)).sum(),
        "Manquant / invalide": df[colonne].isna().sum(),
    })
    repartition = repartition[repartition > 0]
    if repartition.empty:
        ax.text(0.5, 0.5, "Aucune donnée", ha="center", va="center")
    else:
        ax.pie(repartition, labels=repartition.index, autopct="%1.1f%%", startangle=90)
    ax.set_title(colonne)
plt.tight_layout()
plt.show()

In [4]:
%pip install matplotlib numpy pandas

  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl.metadata (142 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl (2.5 MB)
Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl (70 kB)
Using cached pyparsing-3.3.3-py3-none-any.whl (126 kB)



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
